# ArtDiffuse-RL: Kaggle Training

Train a small diffusion model on Midjourney text-to-image data with 2x T4 GPUs.
All code is embedded directly in this notebook (no external dependencies).

In [ ]:
import sys, os, math, time, json
from pathlib import Path
from io import BytesIO

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from PIL import Image
import torchvision.transforms as T
import numpy as np
import yaml
from tqdm import tqdm

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
print(f'GPUs: {torch.cuda.device_count()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

In [ ]:
# ============================================================
# MODEL ARCHITECTURE
# ============================================================

class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device, dtype=torch.float32) / half)
        args = t[:, None].float() * freqs[None]
        return torch.cat([torch.cos(args), torch.sin(args)], dim=-1)


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim, dropout=0.1):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm2 = nn.GroupNorm(8, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.time_proj = nn.Linear(time_dim, out_ch)
        self.dropout = nn.Dropout(dropout)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, t_emb):
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.time_proj(t_emb)[:, :, None, None]
        h = self.conv2(self.dropout(F.silu(self.norm2(h))))
        return h + self.skip(x)


class AttentionBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.norm = nn.GroupNorm(8, ch)
        self.qkv = nn.Conv2d(ch, ch * 3, 1)
        self.proj = nn.Conv2d(ch, ch, 1)

    def forward(self, x):
        b, c, h, w = x.shape
        qkv = self.qkv(self.norm(x)).reshape(b, 3, c, h * w)
        q, k, v = qkv[:, 0], qkv[:, 1], qkv[:, 2]
        attn = torch.softmax(q @ k.transpose(-2, -1) / math.sqrt(c), dim=-1)
        out = (attn @ v).reshape(b, c, h, w)
        return x + self.proj(out)


class Downsample(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv = nn.Conv2d(ch, ch, 3, stride=2, padding=1)

    def forward(self, x):
        return self.conv(x)


class Upsample(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv = nn.Conv2d(ch, ch, 3, padding=1)

    def forward(self, x):
        x = F.interpolate(x, scale_factor=2, mode='nearest')
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=3, ch=128, ch_mult=(1, 2, 4, 4), num_res_blocks=2, dropout=0.1):
        super().__init__()
        self.time_dim = ch * 4
        self.num_res_blocks = num_res_blocks
        self.ch_mult = ch_mult
        self.time_mlp = nn.Sequential(
            SinusoidalTimeEmbedding(ch),
            nn.Linear(ch, self.time_dim),
            nn.SiLU(),
            nn.Linear(self.time_dim, self.time_dim),
        )
        self.conv_in = nn.Conv2d(in_ch, ch, 3, padding=1)

        # Down
        self.down_blocks = nn.ModuleList()
        self.down_samples = nn.ModuleList()
        curr_ch = ch
        for i, mult in enumerate(ch_mult):
            level_ch = ch * mult
            for _ in range(num_res_blocks):
                self.down_blocks.append(ResBlock(curr_ch, level_ch, self.time_dim, dropout))
                curr_ch = level_ch
            if i < len(ch_mult) - 1:
                self.down_samples.append(Downsample(curr_ch))

        # Mid
        self.mid_block1 = ResBlock(curr_ch, curr_ch, self.time_dim, dropout)
        self.mid_attn = AttentionBlock(curr_ch)
        self.mid_block2 = ResBlock(curr_ch, curr_ch, self.time_dim, dropout)

        # Up
        self.up_blocks = nn.ModuleList()
        self.up_samples = nn.ModuleList()
        for i, mult in reversed(list(enumerate(ch_mult))):
            level_ch = ch * mult
            for _ in range(num_res_blocks):
                self.up_blocks.append(ResBlock(curr_ch + level_ch, level_ch, self.time_dim, dropout))
                curr_ch = level_ch
            self.up_blocks.append(ResBlock(curr_ch, level_ch, self.time_dim, dropout))
            curr_ch = level_ch
            if i > 0:
                self.up_samples.append(Upsample(curr_ch))

        self.conv_out = nn.Sequential(
            nn.GroupNorm(8, curr_ch),
            nn.SiLU(),
            nn.Conv2d(curr_ch, out_ch, 3, padding=1),
        )

    def forward(self, x, t):
        t_emb = self.time_mlp(t)
        h = self.conv_in(x)
        skips = []
        block_idx = 0
        for i in range(len(self.ch_mult)):
            for _ in range(self.num_res_blocks):
                h = self.down_blocks[block_idx](h, t_emb)
                skips.append(h)
                block_idx += 1
            if i < len(self.ch_mult) - 1:
                h = self.down_samples[i](h)
        h = self.mid_block1(h, t_emb)
        h = self.mid_attn(h)
        h = self.mid_block2(h, t_emb)
        up_idx = 0
        for i, sample in enumerate(self.up_samples):
            for _ in range(self.num_res_blocks):
                skip = skips.pop()
                h = torch.cat([h, skip], dim=1)
                h = self.up_blocks[up_idx](h, t_emb)
                up_idx += 1
            h = self.up_blocks[up_idx](h, t_emb)
            up_idx += 1
            h = sample(h)
        for _ in range(self.num_res_blocks):
            skip = skips.pop()
            h = torch.cat([h, skip], dim=1)
            h = self.up_blocks[up_idx](h, t_emb)
            up_idx += 1
        h = self.up_blocks[up_idx](h, t_emb)
        return self.conv_out(h)


class DDPM(nn.Module):
    def __init__(self, model, timesteps=1000, beta_start=1e-4, beta_end=2e-2):
        super().__init__()
        self.model = model
        self.timesteps = timesteps
        betas = torch.linspace(beta_start, beta_end, timesteps, dtype=torch.float32)
        alphas = 1.0 - betas
        alphas_cumprod = torch.cumprod(alphas, dim=0)
        alphas_cumprod_prev = F.pad(alphas_cumprod[:-1], (1, 0), value=1.0)
        self.register_buffer('betas', betas)
        self.register_buffer('alphas_cumprod', alphas_cumprod)
        self.register_buffer('alphas_cumprod_prev', alphas_cumprod_prev)
        self.register_buffer('sqrt_alphas_cumprod', torch.sqrt(alphas_cumprod))
        self.register_buffer('sqrt_one_minus_alphas_cumprod', torch.sqrt(1.0 - alphas_cumprod))
        self.register_buffer('sqrt_recip_alphas_cumprod', torch.sqrt(1.0 / alphas_cumprod))
        self.register_buffer('sqrt_recipm1_alphas_cumprod', torch.sqrt(1.0 / alphas_cumprod - 1))
        self.register_buffer('posterior_variance', betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod))

    def q_sample(self, x0, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x0)
        sqrt_ac = self.sqrt_alphas_cumprod[t][:, None, None, None]
        sqrt_omac = self.sqrt_one_minus_alphas_cumprod[t][:, None, None, None]
        return sqrt_ac * x0 + sqrt_omac * noise

    def p_sample(self, x, t):
        eps = self.model(x, t)
        sqrt_recip_ac = self.sqrt_recip_alphas_cumprod[t][:, None, None, None]
        sqrt_recipm1_ac = self.sqrt_recipm1_alphas_cumprod[t][:, None, None, None]
        mean = sqrt_recip_ac * (x - sqrt_recipm1_ac * eps)
        var = self.posterior_variance[t][:, None, None, None]
        noise = torch.randn_like(x)
        nonzero = (t != 0)[:, None, None, None].float()
        return mean + nonzero * torch.sqrt(var) * noise

    def sample(self, shape, device):
        x = torch.randn(shape, device=device)
        for t in reversed(range(self.timesteps)):
            t_batch = torch.full((shape[0],), t, device=device, dtype=torch.long)
            x = self.p_sample(x, t_batch)
        return x

    def compute_loss(self, x0):
        b = x0.shape[0]
        t = torch.randint(0, self.timesteps, (b,), device=x0.device)
        noise = torch.randn_like(x0)
        xt = self.q_sample(x0, t, noise)
        eps_pred = self.model(xt, t)
        return F.mse_loss(eps_pred, noise)

print('Model architecture loaded')

In [ ]:
# ============================================================
# DATASET
# ============================================================

class MidjourneyDataset(Dataset):
    def __init__(self, data_dir, image_size=64, max_samples=50000):
        self.data_dir = Path(data_dir)
        self.image_size = image_size
        self.max_samples = max_samples
        self.transform = T.Compose([
            T.Resize((image_size, image_size)),
            T.RandomHorizontalFlip(),
            T.ToTensor(),
            T.Normalize([0.5], [0.5]),
        ])
        self.data = []
        for json_file in sorted(self.data_dir.glob('*.json')):
            with open(json_file) as f:
                entries = json.load(f)
                for entry in entries:
                    if isinstance(entry, dict):
                        prompt = entry.get('prompt', entry.get('text', ''))
                        image_url = entry.get('image_url', entry.get('url', ''))
                        if prompt and image_url:
                            self.data.append({'prompt': prompt, 'image_url': image_url})
                    if len(self.data) >= max_samples:
                        break
            if len(self.data) >= max_samples:
                break
        print(f'Loaded {len(self.data)} samples')

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        entry = self.data[idx]
        try:
            import requests
            response = requests.get(entry['image_url'], timeout=10)
            image = Image.open(BytesIO(response.content)).convert('RGB')
            image = self.transform(image)
        except Exception as e:
            image = torch.zeros(3, self.image_size, self.image_size)
        return {'image': image, 'caption': entry['prompt']}

print('Dataset class loaded')

In [ ]:
# ============================================================
# TRAINING
# ============================================================

class EMA:
    def __init__(self, model, decay=0.9999):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    def update(self, model):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)
            else:
                self.shadow[k].copy_(v)

    def state_dict(self):
        return self.shadow


def cosine_lr(step, total_steps, warmup=1000, base_lr=1e-4):
    if step < warmup:
        return base_lr * step / warmup
    progress = (step - warmup) / (total_steps - warmup)
    return base_lr * 0.5 * (1 + math.cos(math.pi * progress))


def train(model, dataloader, config, device, output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    lr = config.get('lr', 1e-4)
    weight_decay = config.get('weight_decay', 0.01)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    ema = EMA(model, decay=config.get('ema_decay', 0.9999))
    scaler = GradScaler()
    amp = config.get('amp', True)
    epochs = config.get('epochs', 100)
    grad_accum = config.get('grad_accum', 1)
    save_every = config.get('save_every', 5000)
    log_every = config.get('log_every', 100)
    max_steps = config.get('max_steps', None)
    global_step = 0
    total_steps = epochs * len(dataloader) if max_steps is None else max_steps
    model.train()
    for epoch in range(epochs):
        pbar = tqdm(dataloader, desc=f'Epoch {epoch+1}/{epochs}')
        for batch_idx, batch in enumerate(pbar):
            images = batch['image'].to(device)
            lr_now = cosine_lr(global_step, total_steps, base_lr=lr)
            for g in optimizer.param_groups:
                g['lr'] = lr_now
            with autocast(enabled=amp):
                loss = model.compute_loss(images)
                loss = loss / grad_accum
            scaler.scale(loss).backward()
            if (batch_idx + 1) % grad_accum == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                ema.update(model)
                global_step += 1
            if global_step % log_every == 0:
                pbar.set_postfix(loss=loss.item() * grad_accum, lr=f'{lr_now:.2e}', step=global_step)
            if global_step % save_every == 0:
                ckpt_path = output_dir / f'checkpoint_{global_step}.pt'
                torch.save({'model': model.state_dict(), 'ema': ema.state_dict(), 'optimizer': optimizer.state_dict(), 'step': global_step, 'config': config}, ckpt_path)
                print(f'Saved checkpoint to {ckpt_path}')
            if max_steps and global_step >= max_steps:
                break
        if max_steps and global_step >= max_steps:
            break
    final_path = output_dir / 'final.pt'
    torch.save({'model': model.state_dict(), 'ema': ema.state_dict(), 'optimizer': optimizer.state_dict(), 'step': global_step, 'config': config}, final_path)
    print(f'Training complete. Final model saved to {final_path}')
    return model

print('Training functions loaded')


In [ ]:
# ============================================================
# CONFIG
# ============================================================

config = {
    'model': {
        'in_ch': 3, 'out_ch': 3, 'ch': 128,
        'ch_mult': [1, 2, 4, 4], 'num_res_blocks': 2,
        'dropout': 0.1, 'timesteps': 1000,
        'beta_start': 0.0001, 'beta_end': 0.02,
    },
    'training': {
        'epochs': 100, 'batch_size': 256, 'lr': 0.0001,
        'weight_decay': 0.01, 'ema_decay': 0.9999,
        'amp': True, 'grad_accum': 1,
        'save_every': 5000, 'log_every': 100, 'max_steps': 30000,
    },
    'data': {'dataset': 'midjourney', 'image_size': 64, 'num_workers': 4},
}

print('Config loaded')
for k, v in config.items():
    print(f'  {k}: {v}')

In [ ]:
# ============================================================
# BUILD MODEL
# ============================================================

device = torch.device('cuda')
model = UNet(
    in_ch=config['model']['in_ch'],
    out_ch=config['model']['out_ch'],
    ch=config['model']['ch'],
    ch_mult=tuple(config['model']['ch_mult']),
    num_res_blocks=config['model']['num_res_blocks'],
    dropout=config['model']['dropout'],
)
ddpm = DDPM(
    model,
    timesteps=config['model']['timesteps'],
    beta_start=config['model']['beta_start'],
    beta_end=config['model']['beta_end'],
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {total_params:,}')
print(f'Model size: {total_params * 4 / 1024 / 1024:.1f} MB')

In [ ]:
# ============================================================
# CREATE DATASET
# ============================================================

dataset = MidjourneyDataset(
    data_dir='/kaggle/input/midjourney-texttoimage',
    image_size=64,
    max_samples=50000,
)

dataloader = DataLoader(
    dataset,
    batch_size=config['training']['batch_size'],
    shuffle=True,
    num_workers=config['data']['num_workers'],
    pin_memory=True,
)

print(f'Dataset: {len(dataset)} samples')
print(f'Batches: {len(dataloader)}')

In [ ]:
# ============================================================
# TRAIN
# ============================================================

print('Starting training...')
ddpm = train(
    model=ddpm,
    dataloader=dataloader,
    config=config['training'],
    device=device,
    output_dir='/kaggle/working/models',
)

In [ ]:
# ============================================================
# GENERATE SAMPLES
# ============================================================

from torchvision.utils import save_image

ddpm.eval()
with torch.no_grad():
    samples = ddpm.sample((16, 3, 64, 64), device)

save_image(samples, '/kaggle/working/results/samples.png', nrow=4, normalize=True)
print('Training complete! Samples saved to /kaggle/working/results/samples.png')